# Automated GD Modelling POC

## Day 1 — Environment and ReplicatorAgent setup

This notebook records the reproducible setup and smoke tests for the ReplicatorBench-based proof of concept.

Pinned ReplicatorBench commit:

`fb6a804fd710764f3ad3c8b84e1323c2804c4776`


In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

POC_ROOT = Path.cwd().resolve()
REPO_ROOT = POC_ROOT.parent.parent
REPLICATORBENCH_ROOT = Path(
    os.environ.get(
        "REPLICATORBENCH_DIR",
        REPO_ROOT / "replicatoragent" / "replicatorbench",
    )
).resolve()

# Auto-approve ReplicatorBench's human-confirmation prompts (core/tools.py,
# generator/execute_tools.py) so headless `make` runs below never block on
# stdin. This sets PYTHONPATH so Python auto-imports
# scripts/autoapprove/sitecustomize.py, which monkey-patches `input()`; every
# subprocess.run(["make", ...]) call in this notebook inherits os.environ by
# default, so this only needs to be set once, here.
AUTOAPPROVE_DIR = str(REPO_ROOT / "execution" / "scripts" / "autoapprove")
os.environ["PYTHONPATH"] = (
    AUTOAPPROVE_DIR + os.pathsep + os.environ["PYTHONPATH"]
    if os.environ.get("PYTHONPATH")
    else AUTOAPPROVE_DIR
)

print("POC repository:", POC_ROOT)
print("ReplicatorBench:", REPLICATORBENCH_ROOT)
print("Python:", sys.version)
print("Platform:", platform.platform())

In [2]:
expected_commit = "fb6a804fd710764f3ad3c8b84e1323c2804c4776"

actual_commit = subprocess.run(
    ["git", "-C", str(REPLICATORBENCH_ROOT.parent), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

assert actual_commit == expected_commit, (
    f"Unexpected ReplicatorBench commit: {actual_commit}"
)

print("Pinned commit verified:", actual_commit)


Pinned commit verified: fb6a804fd710764f3ad3c8b84e1323c2804c4776


In [3]:
subprocess.run(
    ["docker", "info"],
    check=True,
    stdout=subprocess.DEVNULL,
)

print("Docker daemon: reachable")


Docker daemon: reachable


In [4]:
result = subprocess.run(
    ["make", "check-deps"],
    cwd=REPLICATORBENCH_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print(result.stdout)


python3 core/check_deps.py pytest pytest_cov openai dotenv pymupdf pyreadr pandas numpy docker docx
All required imports available.



In [5]:
result = subprocess.run(
    ["make", "check-docker"],
    cwd=REPLICATORBENCH_ROOT,
    check=True,
    capture_output=True,
    text=True,
)

print(result.stdout)


Docker: OK



In [ ]:
import subprocess

command = [
    "make",
    "extract-stage1",
    "STUDY=./data/original/1/input",
    "MODEL=gpt-5.4-mini",
]

result = subprocess.run(
    command,
    cwd=REPLICATORBENCH_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

assert result.returncode == 0

The first run used ./data/original/1 and returned code 0 despite reading no study content, producing an all-“not stated” JSON. After correcting the study path to ./data/original/1/input, the extractor produced substantive claim, data, method, result, and metadata fields. This confirms the extraction stage works, while also exposing a silent-failure risk when the study path is incorrect.

In [ ]:
import subprocess

command = [
    "make",
    "pipeline-easy",
    "STUDY=./data/original/1/input",
    "MODEL=gpt-5.4-mini",
]

result = subprocess.run(
    command,
    cwd=REPLICATORBENCH_ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

assert result.returncode == 0